# 250 · electrode coordinates on fsaverage

The **single canonical source** of derived electrode positions for this project: every
brain render and the LanA atlas sampling read what this notebook writes. It is
regenerable pipeline output, not raw data.

**How to run:** from `02_FBM_Clustering`, *Run All*. Cell 1 sets up everything and
prints a **preflight table** — per patient, what is on disk and which route step 2 will
take — before anything is written. Then step 1, step 2, and (optionally) step 3.
Every cell takes what it needs from cell 1, so a cell never fails for want of an import
made somewhere else (the `os` error of 251).

| step | does | writes |
|---|---|---|
| 1 | LEPTOVOX voxel contacts → subject tkrRAS, with a mosaic | `<pid>/glassbrain/coords/<pid>_contacts_tkrRAS.csv`, `…/png/<pid>_mosaic_LEPTOVOX.png`, `_step1_tkrRAS_status.tsv` |
| 2 | tkrRAS → fsaverage: cortical contacts via the subject's spherical registration, white-matter contacts via the talairach affine; Yeo labels | `fsaverage/coords/<pid>_contacts_fsaverage.csv`, `ALL_PATIENTS_contacts_fsaverage.csv`, `…_nowm.csv`, `_regen_qc.tsv`, `fsaverage/png/…mosaic.png` |
| 3 | tkrRAS → Talairach (optional; nothing in the paper reads it) | `talairach/coords/*.csv`, mosaics |

**The fallback.** A subject with no `surf/lh.sphere.reg` / `rh.sphere.reg` (EL046, EL048
as of September 2026) has no surface pathway, so step 2 routes **every** contact through
the talairach affine. Each contact records its route in the `projection` column and the
QC table says `affine-only (no sphere.reg)` for the patient. If the files appear later,
the spherical route is used automatically.

**The LEPTOVOX convention** (determined by Lora, see 251 for the permutation/flip
exploration): coordinates as-is, k axis flipped — the superior/inferior slice direction
is inverted in LEPTOVOX. Index base (0 or 1) is detected per patient.

**Then:** `04_FBM_Pooling/make_lana_runs.py` (samples the atlas at the new coordinates),
notebook 252 with `RUN_FILTER = [{'method': 'atlas'}]` (exports the tables), and the
paper figures.

### Per-patient folder structure the pipeline expects (PAT_* on the share; EL* in the Bern tree, lowercase)
```
PAT_XXXX/
  BIDS/sub-XXXX_electrodes.tsv        white-matter labels (is_wm), via lf_io_utils
  elec_recon/PAT_XXXX.electrodeNames  contact names, two header lines
  elec_recon/PAT_XXXX.LEPTOVOX        contact voxel coordinates
  mri/brainmask.mgz | T1.mgz | orig.mgz | aseg.mgz | ...   any one, for the geometry
  mri/transforms/talairach.xfm        (FastSurfer: mri/transform/talairach.xfm)
  surf/lh.pial  rh.pial               required
  surf/lh.sphere.reg  rh.sphere.reg   spherical route; absent -> affine-only
```


In [1]:
# ============================================================
# 250_recon_fsaverage - CONFIG + PREFLIGHT
#
# Run this cell first; every later cell takes its imports, paths, patient list and
# helpers from here and nothing else. The preflight at the end lists, per patient,
# what is on disk and which route step 2 will take. Nothing is written by this cell.
# ============================================================
import os, re, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.freesurfer.io import read_geometry, read_annot
from scipy.spatial import cKDTree
import pyvista as pv
import imageio.v3 as iio

# ---- where things are ---------------------------------------------------------
#   SHARED_ROOT     : the collaborators' share, READ-ONLY. PAT_* FreeSurfer folders
#                     and the shared fsaverage live here.
#   BERN_RECON_ROOT : the Bern reconstruction tree. EL* folders, lowercase names.
#   OUT_ROOT        : everything this notebook writes.
SHARED_ROOT     = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators")
BERN_RECON_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\Reconstruction")
OUT_ROOT        = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon")
FSAVERAGE       = SHARED_ROOT / "fsaverage"

# the project's own modules: lf_recon_shared (the talairach affine path) and the BIDS
# helpers 140 uses for white matter. The notebook is expected to run from
# 02_FBM_Clustering, where 251 always ran from.
HERE = Path.cwd()
assert (HERE / "functions" / "lf_recon_shared.py").is_file(), (
    f"run this notebook from 02_FBM_Clustering - cwd is {HERE}")
for p in (HERE, HERE / "functions", HERE.parent / "01_FBM_Analysis" / "functions"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import lf_recon_shared as rs
try:
    import lf_io_utils as bids
    _HAVE_BIDS = True
except Exception as _e_bids:
    _HAVE_BIDS = False
    print(f"[WARN] BIDS helpers not importable ({_e_bids}). Without them every contact is "
          f"treated as cortical (is_wm = 0). Usually this means `mne` is missing from "
          f"this kernel - install it, or run in the kernel that has it.")

# ---- who ------------------------------------------------------------------------
# PAT_* are discovered on the share; EL* are listed because the Bern tree also holds
# subjects this project does not use. ONLY, when non-empty, restricts a run to those
# patients - handy for one new recon.
PAT_PATIENTS = sorted(p.name for p in SHARED_ROOT.iterdir()
                      if p.is_dir() and p.name.startswith("PAT_"))
EL_PATIENTS = ["EL030", "EL033", "EL034", "EL035", "EL036", "EL037", "EL038",
               "EL039", "EL040", "EL042", "EL043", "EL044", "EL045", "EL046", "EL048"]
ONLY = []                                   # e.g. ["EL046", "EL048"]
PATIENTS = [p for p in PAT_PATIENTS + EL_PATIENTS if not ONLY or p in ONLY]

# ---- step 1 knobs (the LEPTOVOX convention Lora determined - see 251) -------------
PERM  = (0, 1, 2)             # XYZ as-is
FLIPS = (False, False, True)  # flip k only: k' = (dim_z - 1) - k
VIEWS = ("left", "frontal", "right")
WINDOW_SIZE = (1200, 1000)
TRANSPARENT_BG = True
BRAIN_COLOR = "#ead6db"
BRAIN_OPACITY = 0.25
POINT_COLOR = "purple"
POINT_SIZE = 10
POINT_OPACITY = 0.9
OVERWRITE_CSV = True
OVERWRITE_PNG = True

# ---- shared helpers -------------------------------------------------------------
def subj_dir_for(pid: str) -> Path:
    """PAT_xxxx -> SHARED_ROOT/PAT_xxxx ; ELxxx -> BERN_RECON_ROOT/elxxx (lowercase)."""
    pid = str(pid)
    if pid.startswith("PAT_"):
        return SHARED_ROOT / pid
    if pid.upper().startswith("EL"):
        return BERN_RECON_ROOT / pid.lower()
    raise ValueError(f"Unrecognized patient cohort prefix: {pid}")

def _try_paths(parent: Path, *candidates):
    """The first existing file among candidates inside parent, else None."""
    for name in candidates:
        p = parent / name
        if p.is_file():
            return p
    return None

MGZ_CANDIDATES = ["brainmask.mgz", "T1.mgz", "orig.mgz",
                  "aseg.mgz", "aseg.presurf.mgz",
                  "aparc.DKTatlas+aseg.mapped.mgz", "aparc.DKTatlas+aseg.deep.mgz"]

def pick_mgz(subj_dir: Path) -> Path:
    """Only the voxel-to-RAS geometry is read from this volume, and every volume of
    one FreeSurfer/FastSurfer run carries the same geometry (checked on EL046 and
    EL048). The recon-all three come first so nothing that resolves today changes;
    the rest are what a FastSurfer subject has instead."""
    p = _try_paths(subj_dir / "mri", *MGZ_CANDIDATES)
    if p is None:
        raise FileNotFoundError(f"No usable mgz in {subj_dir / 'mri'}")
    return p

def talairach_xfm_for(pid: str):
    """talairach.xfm in mri/transforms/ (recon-all) or mri/transform/ (FastSurfer)."""
    return _try_paths(subj_dir_for(pid) / "mri", "transforms/talairach.xfm",
                      "transform/talairach.xfm")

def has_sphere_reg(pid: str) -> bool:
    """Both spherical registrations present. Without them step 2 has no surface
    pathway and routes every contact through the talairach affine."""
    sd = subj_dir_for(pid) / "surf"
    return (sd / "lh.sphere.reg").is_file() and (sd / "rh.sphere.reg").is_file()

# ---- PREFLIGHT ------------------------------------------------------------------
def preflight(patients):
    rows = []
    for pid in patients:
        sd = subj_dir_for(pid)
        ed = sd / "elec_recon"
        mgz = _try_paths(sd / "mri", *MGZ_CANDIDATES)
        row = dict(
            patient=pid,
            folder=sd.is_dir(),
            electrodeNames=_try_paths(ed, f"{pid}.electrodeNames", f"{pid.lower()}.electrodeNames") is not None,
            LEPTOVOX=_try_paths(ed, f"{pid}.LEPTOVOX", f"{pid.lower()}.LEPTOVOX") is not None,
            mgz=mgz.name if mgz else "-",
            pial=(sd / "surf" / "lh.pial").is_file() and (sd / "surf" / "rh.pial").is_file(),
            sphere_reg=has_sphere_reg(pid),
            talairach_xfm=talairach_xfm_for(pid) is not None,
            tkrRAS_done=(OUT_ROOT / pid / "glassbrain" / "coords" / f"{pid}_contacts_tkrRAS.csv").is_file(),
        )
        row["step1"] = "ready" if (row["folder"] and row["electrodeNames"] and row["LEPTOVOX"]
                                   and mgz is not None and row["pial"]) else "BLOCKED"
        row["step2_route"] = ("spherical" if row["sphere_reg"] else "affine-only") \
            if row["talairach_xfm"] else "BLOCKED (no talairach.xfm)"
        rows.append(row)
    return pd.DataFrame(rows)

PRE = preflight(PATIENTS)
with pd.option_context("display.width", 200, "display.max_rows", 100):
    print(PRE.to_string(index=False))
print(f"\n{len(PATIENTS)} patients: step 1 ready {int((PRE.step1 == 'ready').sum())}, "
      f"step 2 spherical {int((PRE.step2_route == 'spherical').sum())}, "
      f"affine-only {int((PRE.step2_route == 'affine-only').sum())}, "
      f"blocked {int(PRE.step2_route.str.startswith('BLOCKED').sum())}")
bad = PRE[(PRE.step1 != "ready") | PRE.step2_route.str.startswith("BLOCKED")]
if len(bad):
    print("needs attention:", ", ".join(bad.patient))


 patient  folder  electrodeNames  LEPTOVOX           mgz  pial  sphere_reg  talairach_xfm  tkrRAS_done step1 step2_route
PAT_1145    True            True      True brainmask.mgz  True        True           True         True ready   spherical
PAT_2462    True            True      True brainmask.mgz  True        True           True         True ready   spherical
PAT_2856    True            True      True brainmask.mgz  True        True           True         True ready   spherical
PAT_2868    True            True      True brainmask.mgz  True        True           True         True ready   spherical
PAT_2893    True            True      True brainmask.mgz  True        True           True         True ready   spherical
PAT_3066    True            True      True brainmask.mgz  True        True           True         True ready   spherical
PAT_3301    True            True      True brainmask.mgz  True        True           True         True ready   spherical
PAT_3390    True            True

In [2]:
# ============================================================
# STEP 1 - LEPTOVOX -> tkrRAS, per patient            (251 cell 2, re-homed)
#
# The LEPTOVOX convention is fixed in the config: perm (0,1,2), flip k only.
# For every patient in PATIENTS:
#   OUT_ROOT/<pid>/glassbrain/coords/<pid>_contacts_tkrRAS.csv
#   OUT_ROOT/<pid>/glassbrain/png/<pid>_mosaic_LEPTOVOX.png
# and a status table, OUT_ROOT/_step1_tkrRAS_status.tsv. One patient failing does not
# stop the others.
# ============================================================

# Helpers: 
#---------
def read_electrode_lines_drop2(path: Path) -> list[str]:
    lines = [ln.strip() for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines() if ln.strip()]
    if len(lines) < 3:
        raise ValueError(f"electrodeNames too short: {path}")
    return lines[2:]  # drop timestamp + header

def parse_name_and_hemi(lines: list[str]):
    """
    lines look like: 'FPG1 D L'
    returns:
      name_raw (full line), name (first token), hemi_expected (L/R/None)
    """
    names_raw, names_clean, hemi = [], [], []
    for ln in lines:
        parts = ln.split()
        nm = parts[0] if len(parts) else ln
        h = None
        if len(parts):
            last = parts[-1].upper()
            if last in ("L", "R"):
                h = last
        names_raw.append(ln)
        names_clean.append(nm)
        hemi.append(h)
    return np.array(names_raw, dtype=object), np.array(names_clean, dtype=object), np.array(hemi, dtype=object)

def read_leptovox_xyz(path: Path) -> np.ndarray:
    rows = []
    for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        t = ln.strip()
        if not t or t.startswith("#"):
            continue
        parts = t.replace(",", " ").split()
        if len(parts) < 3:
            continue
        try:
            rows.append((float(parts[0]), float(parts[1]), float(parts[2])))
        except ValueError:
            continue
    if not rows:
        raise ValueError(f"No numeric rows found in {path}")
    return np.asarray(rows, dtype=float)

# Geometry helpers
# ----------------
def voxel_to_tkr(points_ijk: np.ndarray, vox2ras_tkr: np.ndarray) -> np.ndarray:
    n = points_ijk.shape[0]
    ijk_h = np.c_[points_ijk, np.ones(n)]
    tkr_h = (vox2ras_tkr @ ijk_h.T).T
    return tkr_h[:, :3]

def apply_voxel_flips(ijk: np.ndarray, vol_shape, flips=(False, False, True)) -> np.ndarray:
    dims = np.array(vol_shape, dtype=float)
    out = ijk.copy()
    for ax, do_flip in enumerate(flips):
        if do_flip:
            out[:, ax] = (dims[ax] - 1.0) - out[:, ax]
    return out

def nearest_pial_metrics(points_tkr: np.ndarray, lh_v: np.ndarray, rh_v: np.ndarray):
    kdl = cKDTree(lh_v)
    kdr = cKDTree(rh_v)
    dl, _ = kdl.query(points_tkr, k=1, workers=-1)
    dr, _ = kdr.query(points_tkr, k=1, workers=-1)
    is_left = dl <= dr
    dist = np.minimum(dl, dr)
    return dist, is_left



# Rendering helpers
# -------------------------
def make_mesh(v, f):
    return pv.PolyData(v, np.c_[np.full(len(f), 3), f].astype(np.int64))

def set_view(pl, view):
    view = view.lower()
    if view == "left":
        pl.view_yz(negative=True)
    elif view == "frontal":
        pl.view_xz(negative=False)
    elif view == "right":
        pl.view_yz(negative=False)
    else:
        raise ValueError(view)
    pl.camera.zoom(1.15)

def render_view(lh_mesh, rh_mesh, points_tkr, view):
    pl = pv.Plotter(off_screen=True, window_size=WINDOW_SIZE)
    pl.set_background("white")
    pl.add_mesh(lh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
    pl.add_mesh(rh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
    pl.add_points(points_tkr, color=POINT_COLOR, render_points_as_spheres=True,
                  point_size=POINT_SIZE, opacity=POINT_OPACITY)
    set_view(pl, view)
    img = pl.screenshot(transparent_background=TRANSPARENT_BG, return_img=True)
    pl.close()
    return img

def stitch_horiz(imgs):
    H = min(im.shape[0] for im in imgs)
    imgs = [im[:H] for im in imgs]
    return np.concatenate(imgs, axis=1)


# Per patient
# -------------------------
def export_and_mosaic_patient(pid: str):
    pid = str(pid)
    subj_dir = subj_dir_for(pid)
    elec_dir = subj_dir / "elec_recon"

    # Try uppercase and lowercase pid in filename — Bern's convention varies.
    names_path = _try_paths(elec_dir,
                            f"{pid}.electrodeNames", f"{pid.lower()}.electrodeNames")
    vox_path   = _try_paths(elec_dir,
                            f"{pid}.LEPTOVOX",       f"{pid.lower()}.LEPTOVOX")
    if names_path is None:
        raise FileNotFoundError(f"{pid}: missing electrodeNames in {elec_dir}")
    if vox_path is None:
        raise FileNotFoundError(f"{pid}: missing LEPTOVOX in {elec_dir}")

    mgz = pick_mgz(subj_dir)
    img = nib.load(str(mgz))
    vox2ras_tkr = img.header.get_vox2ras_tkr()
    vol_shape = img.shape[:3]

    # surfaces
    lh_v, lh_f = read_geometry(str(subj_dir / "surf" / "lh.pial"))
    rh_v, rh_f = read_geometry(str(subj_dir / "surf" / "rh.pial"))
    lh_mesh = make_mesh(lh_v, lh_f)
    rh_mesh = make_mesh(rh_v, rh_f)

    # names
    lines = read_electrode_lines_drop2(names_path)
    name_raw, name_clean, hemi_expected = parse_name_and_hemi(lines)

    # leptovox coords
    pts = read_leptovox_xyz(vox_path)
    if pts.shape[0] != len(name_raw):
        raise RuntimeError(f"{pid}: LEPTOVOX rows ({pts.shape[0]}) != electrodeNames contacts ({len(name_raw)})")

    # 0/1-based detection
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    dims = np.array(vol_shape, dtype=float)

    looks_one_based  = (mins >= 1).all() and (maxs <= dims).all()
    looks_zero_based = (mins >= 0).all() and (maxs <  dims).all()

    pts_ijk = pts.copy()
    index_mode = "0-based (assumed)"
    if looks_one_based and not looks_zero_based:
        pts_ijk -= 1.0
        index_mode = "1-based->0-based"

    # apply fixed perm + flips
    pts_ijk = pts_ijk[:, PERM]  # (0,1,2) = no-op, kept for explicitness
    pts_ijk = apply_voxel_flips(pts_ijk, vol_shape, FLIPS)  # flip k only

    # voxel -> tkrRAS
    pts_tkr = voxel_to_tkr(pts_ijk, vox2ras_tkr)

    # dist + hemi prediction
    dist_mm, pred_is_left = nearest_pial_metrics(pts_tkr, lh_v, rh_v)

    # outputs
    out_coords = OUT_ROOT / pid / "glassbrain" / "coords"
    out_pngdir = OUT_ROOT / pid / "glassbrain" / "png"
    out_coords.mkdir(parents=True, exist_ok=True)
    out_pngdir.mkdir(parents=True, exist_ok=True)

    out_csv = out_coords / f"{pid}_contacts_tkrRAS.csv"
    out_png = out_pngdir / f"{pid}_mosaic_LEPTOVOX.png"

    # CSV
    if OVERWRITE_CSV or (not out_csv.is_file()):
        df_out = pd.DataFrame({
            "name_raw": name_raw,
            "name": name_clean,
            "hemi_expected": hemi_expected,
            "x": pts_tkr[:,0], "y": pts_tkr[:,1], "z": pts_tkr[:,2],
            "pred_isLeft": pred_is_left.astype(int),
            "dist_to_pial_mm": np.round(dist_mm, 2),
            "source_space": "tkrRAS",
            "source_provenance": f"LEPTOVOX; {index_mode}; perm={PERM}; flips={tuple(int(b) for b in FLIPS)} (flip k only)",
            "leptovox_file": str(vox_path),
            "mgz_used": str(mgz),
        })
        df_out.to_csv(out_csv, index=False)

    # Mosaic
    if OVERWRITE_PNG or (not out_png.is_file()):
        imgs = [render_view(lh_mesh, rh_mesh, pts_tkr, v) for v in VIEWS]
        mosaic = stitch_horiz(imgs)
        iio.imwrite(out_png, mosaic)

    # quick stats
    med_dist = float(np.median(dist_mm))
    pct_left = float(np.mean(pred_is_left) * 100.0)

    return {
        "pid": pid,
        "status": "OK",
        "n_contacts": int(len(name_raw)),
        "median_dist_to_pial_mm": med_dist,
        "pct_pred_left": pct_left,
        "index_mode": index_mode,
        "csv": str(out_csv),
        "png": str(out_png),
    }


rows = []
for pid in PATIENTS:
    try:
        r = export_and_mosaic_patient(pid)
        print(f"[{pid}] {r['n_contacts']:>4} contacts  {r['index_mode']:<18} median dist to pial {r['median_dist_to_pial_mm']:.1f} mm")
    except Exception as e:
        r = {"pid": pid, "status": "ERROR", "error": str(e)}
        print(f"[{pid}] ERROR: {e}")
    rows.append(r)

df_status = pd.DataFrame(rows)
(OUT_ROOT / "_step1_tkrRAS_status.tsv").parent.mkdir(parents=True, exist_ok=True)
df_status.to_csv(OUT_ROOT / "_step1_tkrRAS_status.tsv", sep="\t", index=False)
print(f"\n{int((df_status.status == 'OK').sum())} of {len(df_status)} patients OK  "
      f"-> {OUT_ROOT / '_step1_tkrRAS_status.tsv'}")
df_status


[PAT_1145]  142 contacts  0-based (assumed)  median dist to pial 4.1 mm
[PAT_2462]  133 contacts  0-based (assumed)  median dist to pial 4.0 mm
[PAT_2856]  176 contacts  0-based (assumed)  median dist to pial 3.1 mm
[PAT_2868]   63 contacts  0-based (assumed)  median dist to pial 3.5 mm
[PAT_2893]  173 contacts  0-based (assumed)  median dist to pial 3.3 mm
[PAT_3066]  200 contacts  0-based (assumed)  median dist to pial 3.5 mm
[PAT_3301]  202 contacts  0-based (assumed)  median dist to pial 4.0 mm
[PAT_3390]  118 contacts  0-based (assumed)  median dist to pial 3.7 mm
[PAT_3415]  148 contacts  0-based (assumed)  median dist to pial 1.3 mm
[PAT_3455]  116 contacts  0-based (assumed)  median dist to pial 3.3 mm
[PAT_3780]  136 contacts  0-based (assumed)  median dist to pial 2.9 mm
[PAT_3965]  218 contacts  0-based (assumed)  median dist to pial 4.0 mm
[PAT_3975]  212 contacts  0-based (assumed)  median dist to pial 3.6 mm
[PAT_5515]  150 contacts  0-based (assumed)  median dist to pial

,pid,status,n_contacts,median_dist_to_pial_mm,pct_pred_left,index_mode,csv,png
0,PAT_1145,OK,142,4.131500,88.028169,0-based (assumed),\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
1,PAT_2462,OK,133,4.023581,100.000000,0-based (assumed),\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
2,PAT_2856,OK,176,3.078871,94.318182,0-based (assumed),\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
3,PAT_2868,OK,63,3.539747,100.000000,0-based (assumed),\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
4,PAT_2893,OK,173,3.338248,94.219653,0-based (assumed),\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
5,PAT_3066,OK,200,3.530114,65.500000,0-based (assumed),\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
6,PAT_3301,OK,202,4.040088,52.970297,0-based (assumed),\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
7,PAT_3390,OK,118,3.740455,100.000000,0-based (assumed),\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
8,PAT_3415,OK,148,1.312528,100.000000,0-based (assumed),\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...
9,PAT_3455,OK,116,3.311701,0.000000,0-based (assumed),\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...,\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\...


In [3]:
# ============================================================
# STEP 2 - tkrRAS -> fsaverage, per patient, then the group tables   (251 cell 4)
#
# Needs step 1's tkrRAS CSVs. Writes, under OUT_ROOT/fsaverage/coords/:
#   <pid>_contacts_fsaverage.csv, ALL_PATIENTS_contacts_fsaverage.csv (with WM),
#   ALL_PATIENTS_contacts_fsaverage_nowm.csv, _regen_qc.tsv
# and the group mosaic under OUT_ROOT/fsaverage/png/.
#
# Surface-based group mapping: patient → fsaverage
#
# Three things this cell does (per patient):
#   1) Project surface contacts (is_wm == 0):
#        - PAT (HUG, LEPTOVOX): subject pial → spherical registration → fsaverage
#        - EL  (BERN, Lookup):   subject tkrRAS → talairach affine → fsaverage tkrRAS
#        - a subject with NO surf/*.sphere.reg (EL046, EL048): every contact through
#          the talairach affine, recorded per contact in `projection` and per
#          patient in the QC table. Adequate for anything sampled at a coordinate;
#          only the snap onto the fsaverage pial (and the Yeo label with it) is lost.
#   2) Project depth contacts (is_wm == 1):
#        - Always talairach affine (no snap to cortex). Keeps contact at its
#          fsaverage-MNI volume position rather than collapsing to nearest pial
#          vertex — the previous behaviour mis-placed every hippocampal /
#          amygdalar / deep-white-matter contact onto the surface.
#   3) Look up Yeo 2011 functional networks (7 and 17) at the nearest
#      fsaverage pial vertex for cortical contacts. Depth contacts get
#      yeo7_network = yeo17_network = "WhiteMatter".
#
# Outputs (under OUT_ROOT/fsaverage/coords/):
#   {pid}_contacts_fsaverage.csv               (per patient)
#   ALL_PATIENTS_contacts_fsaverage.csv         (with WM contacts)
#   ALL_PATIENTS_contacts_fsaverage_nowm.csv    (is_wm == 0 only — what 252
#                                                reads when KEEP_WM=False)
#
# Schema:
#   patient, cohort, name, name_raw, hemi, x, y, z, is_wm,
#   is_cortical, projection, dist_to_pial_mm,
#   yeo7_network, yeo17_network
# ============================================================

# ------------------------------------------------------------
# LOAD FSAVERAGE SURFACES (used by PAT spherical pathway AND Yeo lookup)
# ------------------------------------------------------------
fs_lh_v, fs_lh_f = read_geometry(str(FSAVERAGE / "surf" / "lh.pial"))
fs_rh_v, fs_rh_f = read_geometry(str(FSAVERAGE / "surf" / "rh.pial"))
fs_lh_sph, _ = read_geometry(str(FSAVERAGE / "surf" / "lh.sphere.reg"))
fs_rh_sph, _ = read_geometry(str(FSAVERAGE / "surf" / "rh.sphere.reg"))
fs_lh_sphkd  = cKDTree(fs_lh_sph)    # spherical (PAT projection)
fs_rh_sphkd  = cKDTree(fs_rh_sph)
fs_lh_pialkd = cKDTree(fs_lh_v)      # pial (Yeo nearest-vertex + dist-to-pial)
fs_rh_pialkd = cKDTree(fs_rh_v)

# ------------------------------------------------------------
# LOAD YEO 2011 ANNOTS (7 + 17 networks) from fsaverage/label/
# ------------------------------------------------------------
def _decode_annot_names(names):
    return [n.decode("utf-8") if isinstance(n, (bytes, bytearray)) else str(n) for n in names]

# Where to look for Yeo 2011 annot files (FreeSurfer ships these by
# default at $FREESURFER_HOME/subjects/fsaverage/label). Search order
# (first hit wins):
#   1. $FREESURFER_HOME/subjects/fsaverage/label/  (canonical FS install)
#   2. $SUBJECTS_DIR/fsaverage/label/              (alternate FS install)
#   3. BERN_RECON_ROOT/fsaverage/label/            (Bern reconstruction
#      tree — the primary internal source on this server)
#   4. OUT_ROOT/fsaverage/label/                   (analysis repo — commit
#      them here if you want them tracked alongside the recon outputs)
#   5. SHARED_ROOT/fsaverage/label/                (#SHARE folder — READ
#      ONLY fallback. Files have been there since 2011; we never write
#      into this directory.)
#
# Explicitly NOT searched: MNE's local fsaverage cache
# (~/mne_data/MNE-fsaverage-data/) — per user policy.
YEO_ANNOT_SEARCH = []
_fs_home = os.environ.get("FREESURFER_HOME")
if _fs_home and _fs_home not in (".", "/", "\\", "S:", "S:\\"):
    # Guard against FREESURFER_HOME being set to just a drive letter — saw
    # that on this Windows server, leads to bogus 'S:\subjects\fsaverage'
    # being probed.
    YEO_ANNOT_SEARCH.append(Path(_fs_home) / "subjects" / "fsaverage" / "label")
_subj_dir = os.environ.get("SUBJECTS_DIR")
if _subj_dir and _subj_dir not in (".", "/", "\\"):
    YEO_ANNOT_SEARCH.append(Path(_subj_dir) / "fsaverage" / "label")
YEO_ANNOT_SEARCH.append(BERN_RECON_ROOT / "fsaverage" / "label")
YEO_ANNOT_SEARCH.append(OUT_ROOT / "fsaverage" / "label")
YEO_ANNOT_SEARCH.append(SHARED_ROOT / "fsaverage" / "label")  # read-only fallback

def _try_load_yeo(n):
    """Returns (lh_labels, rh_labels, lh_names, rh_names) or None if missing.

    Walks YEO_ANNOT_SEARCH; first directory containing BOTH lh./rh. annots
    wins. Never reads from #SHARE/To_send_collaborators (policy).
    """
    suffix = f"Yeo2011_{n}Networks_N1000.annot"
    for cand in YEO_ANNOT_SEARCH:
        lh_p = cand / f"lh.{suffix}"
        rh_p = cand / f"rh.{suffix}"
        if lh_p.exists() and rh_p.exists():
            print(f"[Yeo{n}] loading from {cand}")
            lh_lbl, _, lh_names = read_annot(str(lh_p))
            rh_lbl, _, rh_names = read_annot(str(rh_p))
            return lh_lbl, rh_lbl, _decode_annot_names(lh_names), _decode_annot_names(rh_names)
    print(f"[WARN] Yeo {n}-network annot not found. Searched:")
    for cand in YEO_ANNOT_SEARCH:
        print(f"         {cand}")
    print(f"       FreeSurfer ships these by default — set FREESURFER_HOME or")
    print(f"       drop them into {OUT_ROOT / 'fsaverage' / 'label'} .")
    print(f"       yeo{n}_network column will be 'Unavailable' until then.")
    return None

YEO7  = _try_load_yeo(7)
YEO17 = _try_load_yeo(17)

def _yeo_at(xyz, hemi_norm, annots):
    """Look up Yeo network label at nearest fsaverage pial vertex."""
    if annots is None:
        return "Unavailable"
    lh_lbl, rh_lbl, lh_names, rh_names = annots
    if hemi_norm == "lh":
        _, v = fs_lh_pialkd.query(xyz)
        lid = int(lh_lbl[v])
        return lh_names[lid] if 0 <= lid < len(lh_names) else "unknown"
    else:
        _, v = fs_rh_pialkd.query(xyz)
        lid = int(rh_lbl[v])
        return rh_names[lid] if 0 <= lid < len(rh_names) else "unknown"

def _dist_to_pial_mm(xyz, hemi_norm):
    """Euclidean distance from xyz to the nearest fsaverage pial vertex."""
    if hemi_norm == "lh":
        d, _ = fs_lh_pialkd.query(xyz)
    else:
        d, _ = fs_rh_pialkd.query(xyz)
    return float(d)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def _cohort(pid: str) -> str:
    if pid.startswith("PAT_"): return "HUG"
    if pid.startswith("EL"):    return "BERN"
    if pid.startswith("MicroEPI") or pid.startswith("G-") or pid.startswith("B-"):
        return "MICROEPI"
    return "UNKNOWN"

def _hemi_from_x(x: float) -> str:
    return "L" if x < 0 else "R"

def _hemi_norm(h: str) -> str:
    h = str(h).upper()
    return "lh" if h.startswith("L") else "rh"

def _col(df, idx, col, fallback_col=None):
    if col in df.columns:
        return df.iloc[idx][col]
    if fallback_col and fallback_col in df.columns:
        return df.iloc[idx][fallback_col]
    return ""

def make_mesh(v, f):
    return pv.PolyData(v, np.c_[np.full(len(f), 3), f].astype(np.int64))

def stitch_horiz(imgs):
    H = min(im.shape[0] for im in imgs)
    return np.concatenate([im[:H] for im in imgs], axis=1)

def render_fsaverage(points):
    lh = make_mesh(fs_lh_v, fs_lh_f)
    rh = make_mesh(fs_rh_v, fs_rh_f)
    views = []
    for view in ("left", "frontal", "right"):
        pl = pv.Plotter(off_screen=True, window_size=(1200, 1000))
        pl.set_background("white")
        pl.add_mesh(lh, color="#ead6db", opacity=0.25)
        pl.add_mesh(rh, color="#ead6db", opacity=0.25)
        pl.add_points(points, color="purple", point_size=10,
                      render_points_as_spheres=True)
        if   view == "left":    pl.view_yz(negative=True)
        elif view == "right":   pl.view_yz(negative=False)
        else:                    pl.view_xz()
        img = pl.screenshot(transparent_background=True, return_img=True)
        pl.close()
        views.append(img)
    return stitch_horiz(views)

# ------------------------------------------------------------
# PROJECTION HELPERS
# ------------------------------------------------------------
def _pat_surface_proj(pid, pts_surface):
    """LEPTOVOX → subject pial nearest-vertex + spherical reg → fsaverage.

    Works for both PAT (subj_dir under SHARED_ROOT) and EL (subj_dir under
    BERN_RECON_ROOT). Cohort routing lives in subj_dir_for(pid) at the top
    of this cell.
    """
    subj_dir = subj_dir_for(pid)
    lh_v, _   = read_geometry(str(subj_dir / "surf" / "lh.pial"))
    rh_v, _   = read_geometry(str(subj_dir / "surf" / "rh.pial"))
    lh_sph, _ = read_geometry(str(subj_dir / "surf" / "lh.sphere.reg"))
    rh_sph, _ = read_geometry(str(subj_dir / "surf" / "rh.sphere.reg"))
    lh_kd = cKDTree(lh_v); rh_kd = cKDTree(rh_v)

    n = len(pts_surface)
    fs_xyz = np.zeros((n, 3), dtype=float)
    hemi   = ["R"] * n
    for i, p in enumerate(pts_surface):
        dl, il = lh_kd.query(p); dr, ir = rh_kd.query(p)
        if dl <= dr:
            hemi[i] = "L"
            _, fv = fs_lh_sphkd.query(lh_sph[il])
            fs_xyz[i] = fs_lh_v[fv]
        else:
            hemi[i] = "R"
            _, fv = fs_rh_sphkd.query(rh_sph[ir])
            fs_xyz[i] = fs_rh_v[fv]
    return fs_xyz, hemi

def project_patient(pid, df_native):
    """
    Compute fsaverage XYZ + hemi for every contact. Same path for every
    cohort now that both PAT and EL go through LEPTOVOX → tkrRAS.

    Branching is_wm:
      - is_wm == 0 (cortical / grid): spherical registration via subject pial
      - is_wm == 1 (depth white-matter): talairach affine via rs (no surface snap)

    A subject with NO surf/*.sphere.reg (EL046, EL048 as of 2026-09) has no surface
    pathway at all, so EVERY contact goes through the talairach affine, and the
    per-contact `projection` says so. That is adequate for anything sampled at a
    coordinate (the LanA atlas); what is lost for those contacts is the snap onto
    the fsaverage pial, and with it the nearest-vertex Yeo label.

    Returns fs_xyz, hemi, is_wm, projection  ('spherical' or 'affine' per contact).
    """
    pts = df_native[["x", "y", "z"]].to_numpy(float)
    n   = len(pts)
    is_wm = (df_native["is_wm"].to_numpy(int)
             if "is_wm" in df_native.columns
             else np.zeros(n, dtype=int))

    fs_xyz = np.zeros((n, 3), dtype=float)
    hemi   = ["R"] * n
    proj   = np.array(["affine"] * n, dtype=object)

    spherical_ok = has_sphere_reg(pid)
    if not spherical_ok:
        print(f"  [{pid}] no surf/*.sphere.reg - every contact through the talairach "
              f"affine (no pial snap)")

    # depth contacts always; every contact when the surface pathway is unavailable
    affine_idx = np.where(is_wm == 1)[0] if spherical_ok else np.arange(n)
    if len(affine_idx) > 0:
        pts_tal, _ = rs.subject_tkr_to_fsaverage_tkr(pid, pts[affine_idx])
        for j, i in enumerate(affine_idx):
            fs_xyz[i] = pts_tal[j]
            hemi[i]   = _hemi_from_x(float(pts_tal[j, 0]))

    surf_idx = np.where(is_wm == 0)[0] if spherical_ok else np.array([], dtype=int)
    if len(surf_idx) > 0:
        surf_xyz, surf_hemi = _pat_surface_proj(pid, pts[surf_idx])
        for j, i in enumerate(surf_idx):
            fs_xyz[i] = surf_xyz[j]
            hemi[i]   = surf_hemi[j]
            proj[i]   = "spherical"

    return fs_xyz, hemi, is_wm, proj

def _bids_wm_set_for(pid):
    """Set of normalized WM channel names for a patient, derived from the
    BIDS electrodes.tsv (tissueLabel starts with 'wm-'). Returns empty set
    if lf_io_utils isn't importable or the TSV is missing — caller then
    falls back to is_wm=0 for everyone, matching the LEPTOVOX default.
    """
    if not _HAVE_BIDS:
        return set()
    try:
        return bids.wm_labels_for_patient(pid)
    except Exception as _e:
        print(f'  [WARN] {pid}: BIDS WM lookup failed ({_e}); using is_wm=0')
        return set()


def _populate_is_wm_from_bids(pid, df_native):
    """Mutate df_native in-place: set 'is_wm' column from BIDS electrodes.tsv.
    Match contact names case/separator-insensitively (same convention 140 uses).
    Leaves existing is_wm column alone if no BIDS WM names are found.
    """
    wm_set = _bids_wm_set_for(pid)
    if not wm_set:
        # Ensure the column exists with 0s so downstream code is happy.
        if 'is_wm' not in df_native.columns:
            df_native['is_wm'] = 0
        return 0
    name_col = 'name' if 'name' in df_native.columns else 'electrode'
    if not _HAVE_BIDS:
        return 0
    df_native['is_wm'] = df_native[name_col].astype(str).apply(
        lambda n: 1 if bids.normalize_label(n) in wm_set else 0).astype(int)
    return int((df_native['is_wm'] == 1).sum())


def build_patient_df(pid, df_native, cohort_str):
    """Wrap BIDS-is_wm-enrichment + projection + Yeo lookup into a unified df."""
    _n_wm_bids = _populate_is_wm_from_bids(pid, df_native)
    if _n_wm_bids > 0:
        print(f'  [{pid}] BIDS is_wm: {_n_wm_bids} contacts flagged as white-matter')
    fs_xyz, hemi, is_wm, proj = project_patient(pid, df_native)
    rows = []
    for i in range(len(df_native)):
        name = _col(df_native, i, "name", "electrode")
        name_raw = _col(df_native, i, "name_raw") or name
        hn = _hemi_norm(hemi[i])
        is_wm_i = int(is_wm[i])
        is_cortical = 0 if is_wm_i == 1 else 1
        if is_wm_i == 1:
            yeo7  = "WhiteMatter"
            yeo17 = "WhiteMatter"
            dist_pial = float("nan")
        else:
            yeo7  = _yeo_at(fs_xyz[i], hn, YEO7)
            yeo17 = _yeo_at(fs_xyz[i], hn, YEO17)
            dist_pial = _dist_to_pial_mm(fs_xyz[i], hn)
        rows.append({
            "patient":         pid,
            "cohort":          cohort_str,
            "name":            str(name).strip(),
            "name_raw":        str(name_raw).strip(),
            "hemi":            hemi[i],
            "x":               float(fs_xyz[i, 0]),
            "y":               float(fs_xyz[i, 1]),
            "z":               float(fs_xyz[i, 2]),
            "is_wm":           is_wm_i,
            "is_cortical":     is_cortical,
            "projection":      str(proj[i]),
            "dist_to_pial_mm": dist_pial,
            "yeo7_network":    yeo7,
            "yeo17_network":   yeo17,
        })
    return pd.DataFrame(rows)

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
all_rows = []
qc = []

# Unified main loop — every cohort reads its tkrRAS CSV produced by cell 2.
# No more Lookup.xlsx fallback for ELs; the LEPTOVOX path applies to all.
for pid in PATIENTS:
    cohort_str = _cohort(pid)
    try:
        csv_in = OUT_ROOT / pid / "glassbrain" / "coords" / f"{pid}_contacts_tkrRAS.csv"
        if not csv_in.is_file():
            print(f"[{pid}] SKIP: missing {csv_in} — run cell 2 first.")
            qc.append({"patient": pid, "status": "missing-tkrRAS-csv"})
            continue
        df_native = pd.read_csv(csv_in)

        df_fs = build_patient_df(pid, df_native, cohort_str)

        out_dir = OUT_ROOT / "fsaverage" / "coords"
        out_dir.mkdir(parents=True, exist_ok=True)
        df_fs.to_csv(out_dir / f"{pid}_contacts_fsaverage.csv", index=False)
        all_rows.append(df_fs)

        n_total = len(df_fs)
        n_wm    = int((df_fs["is_wm"] == 1).sum())
        n_cort  = int((df_fs["is_cortical"] == 1).sum())
        route   = "spherical" if has_sphere_reg(pid) else "affine-only (no sphere.reg)"
        qc.append({
            "patient": pid, "status": "OK",
            "n_contacts": n_total, "n_cortical": n_cort, "n_wm": n_wm,
            "projection": route,
        })
        print(f"[{pid}] {n_total:>4} contacts  cortical={n_cort:>3}  wm={n_wm:>3}  "
              f"{route:<28} ->  {pid}_contacts_fsaverage.csv")
    except Exception as e:
        qc.append({"patient": pid, "status": f"ERROR: {e}"})
        print(f"[{pid}] ERROR: {e}")

# ------------------------------------------------------------
# AGGREGATES — both variants
# ------------------------------------------------------------
if not all_rows:
    raise RuntimeError("No patients projected successfully — aborting aggregate write.")

df_all = pd.concat(all_rows, ignore_index=True)

out_dir = OUT_ROOT / "fsaverage" / "coords"
out_dir.mkdir(parents=True, exist_ok=True)

out_with_wm = out_dir / "ALL_PATIENTS_contacts_fsaverage.csv"
df_all.to_csv(out_with_wm, index=False)
print(f"\nSaved with-WM aggregate ({len(df_all):>4} rows) -> {out_with_wm.name}")

df_nowm = df_all[df_all["is_wm"] == 0].copy()
out_nowm = out_dir / "ALL_PATIENTS_contacts_fsaverage_nowm.csv"
df_nowm.to_csv(out_nowm, index=False)
print(f"Saved no-WM   aggregate ({len(df_nowm):>4} rows) -> {out_nowm.name}  (252 KEEP_WM=False reads this)")

# QC TSV
pd.DataFrame(qc).to_csv(out_dir / "_regen_qc.tsv", sep="\t", index=False)
print(f"QC table: {(out_dir / '_regen_qc.tsv')}")

# Yeo coverage summary
print("\nYeo 7  distribution:")
print(df_all["yeo7_network"].value_counts())
print("\nYeo 17 distribution (top 20):")
print(df_all["yeo17_network"].value_counts().head(20))

# ------------------------------------------------------------
# GROUP MOSAIC PNG (unchanged behaviour)
# ------------------------------------------------------------
pts_all = df_all[["x", "y", "z"]].to_numpy(float)
mosaic = render_fsaverage(pts_all)
png_dir = OUT_ROOT / "fsaverage" / "png"
png_dir.mkdir(parents=True, exist_ok=True)
png_out = png_dir / "ALL_PATIENTS_fsaverage_mosaic.png"
iio.imwrite(png_out, mosaic)

print(png_dir)
print(f"Saved group fsaverage mosaic -> {png_out}")


[Yeo7] loading from \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\Reconstruction\fsaverage\label
[Yeo17] loading from \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\Reconstruction\fsaverage\label
  [PAT_1145] BIDS is_wm: 38 contacts flagged as white-matter
[PAT_1145]  142 contacts  cortical=104  wm= 38  spherical                    ->  PAT_1145_contacts_fsaverage.csv
[LF 20:34:00] [WM] No electrodes TSV matching: \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\BIDS_elec\SEEG-HUG\sub-2462\ieeg\*_electrodes.tsv
[PAT_2462]  133 contacts  cortical=133  wm=  0  spherical                    ->  PAT_2462_contacts_fsaverage.csv
  [PAT_2856] BIDS is_wm: 37 contacts flagged as white-matter
[PAT_2856]  176 contacts  cortical=139  wm= 37  spherical                    ->  PAT_2856_contacts_fsaverage.csv
  [PAT_2868] BIDS is_wm: 23 contacts flagged as white-matter
[PAT_2868]   63 contacts  cortical= 40  wm= 23  spherical                    ->  PAT_2868_contacts_

In [4]:
# ============================================================
# STEP 3 (optional) - tkrRAS -> Talairach                    (251 cell 6, re-homed)
#
# Volume-space coordinates via each subject's talairach.xfm, for every patient in
# PATIENTS (EL patients included now; the FastSurfer transform/ folder is found too).
# Writes OUT_ROOT/talairach/coords/*.csv and two mosaics. None of the paper figures
# read these - they use the fsaverage tables from step 2.
# ============================================================

PATIENT_COLORS = [
    "blueviolet","fuchsia","deeppink","crimson","pink","red",
    "chocolate","gold","purple","saddlebrown","lemonchiffon",
    "lavenderblush","lime","powderblue","forestgreen","lightcyan",
    "navy","darkslategray","black","darkred","darkolivegreen","aquamarine","aquamarine","aquamarine"
]

def render_volumetric_by_patient(df_all):
    """
    Render Talairach/MNI points with one color per patient
    to visually detect bad transforms.
    """
    pl_views = []

    for view in ("left", "frontal", "right"):
        pl = pv.Plotter(off_screen=True, window_size=(1200, 1000))
        pl.set_background("white")
        if "patient" not in df_all.columns:    # older tables: recover it from the path
            df_all["patient"] = df_all["leptovox_file"].str.extract(r"((?:PAT_|EL)\w+?)[\\/]", expand=False)

        for i, (pid, dfp) in enumerate(df_all.groupby("patient")):
            color = PATIENT_COLORS[i % len(PATIENT_COLORS)]
            pts = dfp[["x", "y", "z"]].to_numpy(float)

            pl.add_points(
                pts,
                color=color,
                point_size=10,
                render_points_as_spheres=True,
                opacity=0.9,
                label=pid,
            )

        if view == "left":
            pl.view_yz(negative=True)
        elif view == "right":
            pl.view_yz(negative=False)
        else:
            pl.view_xz()

        img = pl.screenshot(transparent_background=True, return_img=True)
        pl.close()
        pl_views.append(img)

    # stitch horizontally
    H = min(im.shape[0] for im in pl_views)
    return np.concatenate([im[:H] for im in pl_views], axis=1)


# ------------------------------------------------------------
# TAL TRANSFORM PARSER
# ------------------------------------------------------------
def load_talairach_xfm(xfm_path: Path) -> np.ndarray:
    """
    Robust FreeSurfer talairach.xfm / MNI linear transform parser.
    Handles trailing semicolons and format variants.
    Returns 4x4 affine (RAS -> MNI/Talairach).
    """
    lines = xfm_path.read_text(encoding="utf-8", errors="ignore").splitlines()

    start = None
    for i, ln in enumerate(lines):
        if ln.strip().startswith("Linear_Transform"):
            start = i + 1
            break

    if start is None:
        raise RuntimeError(f"Could not find Linear_Transform in {xfm_path}")

    rows = []
    for j in range(3):
        raw = lines[start + j].strip()
        parts = [p.rstrip(";") for p in raw.split()]
        if len(parts) != 4:
            raise RuntimeError(f"Invalid transform row: {raw}")
        rows.append([float(p) for p in parts])

    M = np.eye(4)
    M[:3, :4] = np.array(rows, dtype=float)
    return M


def apply_affine(points_xyz: np.ndarray, M: np.ndarray) -> np.ndarray:
    n = points_xyz.shape[0]
    xyz_h = np.c_[points_xyz, np.ones(n)]
    out = (M @ xyz_h.T).T
    return out[:, :3]


# ------------------------------------------------------------
# RENDERING
# ------------------------------------------------------------
def render_volumetric(points, color="purple"):
    views = []
    for view in ("left", "frontal", "right"):
        pl = pv.Plotter(off_screen=True, window_size=(1200, 1000))
        pl.set_background("white")
        pl.add_points(points, color=color, point_size=10, render_points_as_spheres=True)

        if view == "left":
            pl.view_yz(negative=True)
        elif view == "right":
            pl.view_yz(negative=False)
        else:
            pl.view_xz()

        img = pl.screenshot(transparent_background=True, return_img=True)
        pl.close()
        views.append(img)

    H = min(im.shape[0] for im in views)
    return np.concatenate([im[:H] for im in views], axis=1)


# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
all_rows = []

for pid in PATIENTS:
    print(f"Talairach mapping: {pid}")
    csv_in = OUT_ROOT / pid / "glassbrain" / "coords" / f"{pid}_contacts_tkrRAS.csv"
    xfm = talairach_xfm_for(pid)                 # transforms/ or transform/

    if not csv_in.is_file():
        print(f"  SKIP: missing {csv_in}")
        continue
    if xfm is None:
        print(f"  SKIP: no talairach.xfm for {pid}")
        continue

    # Load data
    df = pd.read_csv(csv_in)
    pts = df[["x", "y", "z"]].to_numpy(float)

    # Load talairach transform
    M = load_talairach_xfm(xfm)

    # Apply transform
    pts_tal = apply_affine(pts, M)

    # Save per-patient CSV
    out_dir = OUT_ROOT / "talairach" / "coords"
    out_dir.mkdir(parents=True, exist_ok=True)

    df_out = df.copy()
    df_out[["x", "y", "z"]] = pts_tal
    df_out["space"] = "Talairach"
    df_out["transform"] = "FreeSurfer talairach.xfm"
    df_out["patient"] = pid

    out_csv = out_dir / f"{pid}_contacts_talairach.csv"
    df_out.to_csv(out_csv, index=False)

    all_rows.append(df_out)


# # ------------------------------------------------------------
# # CONCATENATE + RENDER GROUP
# # ------------------------------------------------------------
df_all = pd.concat(all_rows, ignore_index=True)

out_all = OUT_ROOT / "talairach" / "coords" / "ALL_PATIENTS_contacts_talairach.csv"
out_all.parent.mkdir(parents=True, exist_ok=True)
df_all.to_csv(out_all, index=False)

print(f"Saved Talairach group CSV → {out_all}")

# Render combined volumetric view
pts_all = df_all[["x", "y", "z"]].to_numpy(float)
mosaic = render_volumetric(pts_all)

png_dir = OUT_ROOT / "talairach" / "png"
png_dir.mkdir(parents=True, exist_ok=True)

png_out = png_dir / "ALL_PATIENTS_talairach_mosaic.png"
iio.imwrite(png_out, mosaic)

print(f"Saved Talairach mosaic → {png_out}")

mosaic = render_volumetric_by_patient(df_all)

png_dir = OUT_ROOT / "talairach" / "png"
png_dir.mkdir(parents=True, exist_ok=True)

png_out = png_dir / "ALL_PATIENTS_talairach_mosaic_colorcoded.png"
iio.imwrite(png_out, mosaic)

print(f"Saved color-coded Talairach mosaic → {png_out}")



Talairach mapping: PAT_1145
Talairach mapping: PAT_2462
Talairach mapping: PAT_2856
Talairach mapping: PAT_2868
Talairach mapping: PAT_2893
Talairach mapping: PAT_3066
Talairach mapping: PAT_3301
Talairach mapping: PAT_3390
Talairach mapping: PAT_3415
Talairach mapping: PAT_3455
Talairach mapping: PAT_3780
Talairach mapping: PAT_3965
Talairach mapping: PAT_3975
Talairach mapping: PAT_5515
Talairach mapping: PAT_5533
Talairach mapping: PAT_648
Talairach mapping: PAT_6619
Talairach mapping: PAT_6684
Talairach mapping: PAT_6704
Talairach mapping: PAT_6739
Talairach mapping: PAT_6854
Talairach mapping: PAT_6953
Talairach mapping: PAT_6980
Talairach mapping: PAT_699
Talairach mapping: PAT_7045
Talairach mapping: EL030
Talairach mapping: EL033
Talairach mapping: EL034
Talairach mapping: EL035
Talairach mapping: EL036
Talairach mapping: EL037
Talairach mapping: EL038
Talairach mapping: EL039
Talairach mapping: EL040
Talairach mapping: EL042
Talairach mapping: EL043
Talairach mapping: EL044
Ta

## What to run next

1. `cd ..\04_FBM_Pooling; python .\make_lana_runs.py; cd ..\02_FBM_Clustering` — resamples the LanA atlas at the coordinates written above and registers six atlas runs.
2. Notebook **252** with `RUN_FILTER = [{'method': 'atlas'}]` in its cell 1 — exports the atlas tables FIG 3 reads.
3. The paper figures: `00_paper2_figure0_coverage.py`, `00_Paper2_Figures.py --figure 1`, the four `00_paper2_figures2_2.py --k 8 --algo-feature-set …`, `00_paper2_figure3_lana.py --all --k 8`.

Notebook **251** is the previous version of this one and is kept for its history; do not run both.
